In [ ]:
import pandas as pd
import numpy as np
from Bio import SeqIO, SeqRecord
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import colorcet

import sys
sys.path.append('./')
import variables as v

In [ ]:
assemblies = pd.read_csv(f'{v.base_dir}script/assemblies.csv').set_index('assembly_name')

# Library read lengths distributions

In [ ]:
RL = {}
for s in v.batches_info['strain'].unique():
    with open(f'/data/mathieu/minknow/combined/{s}.readlengths.txt') as fi:
        rl = fi.read().splitlines()
    RL[s] = rl

In [ ]:
batch_color = dict(zip(range(1,9), colorcet.glasbey_bw))
fig, axes = plt.subplots(ncols=2, figsize=[12,6])

for (b,s), df in v.batches_info.groupby(['batch', 'strain']):
        
    rl = RL[s]
    c = batch_color[b]

    rl = np.sort(np.array(rl).astype(np.int32))
    rl = rl[rl>=200]
    cs = np.cumsum(rl)
    rl_log = np.log10(rl)
    cs_log = np.log10(cs)
    
    ax = axes[0]
    ax.plot(rl_log, np.linspace(0, 100, rl.shape[0]), c=c, lw=0.5)

    ax = axes[1]
    ax.plot(rl_log, cs_log/cs_log.max(), c=c, lw=0.5)

ax = axes[0]
#ax.set_ylim(0, 6)
ax.margins(0.1)
ax.set_xlabel('log10 read length')
ax.set_ylabel('% reads')

ax = axes[1]
#ax.set_ylim(0, 1)
#ax.set_xlim(0, 6)
ax.margins(0.1)
ax.set_xlabel('log10 read length')
ax.set_ylabel('normalized cumul. log10 read length')

legend_elms = [Line2D([0], [0], c=c, label=f'batch{b}') for (b, c) in batch_color.items()]
ax.legend(handles=legend_elms)

plt.show()
plt.close()

# Import assemblies

In [ ]:
Assembly = {}
Tig_Off_Assembly = {}
Coords = {}
Ctg_Assign = {}

for an in assemblies.index:

    asm_path = assemblies.loc[an, 'path']
    assembly = {seq.id:seq for seq in SeqIO.parse(f'{v.base_dir}{asm_path}', 'fasta')}
    Assembly[an] = assembly

    coords_path = f'{v.base_dir}mummer/{an}.S288C.coords'
    coords = pd.read_csv(coords_path, sep='\t', skiprows=4, header=None)
    coords = coords.loc[(coords[4]>5000) & (coords[5]>5000)]
    coords[10] = coords[10].astype(str)
    
    ctg_assign = coords.sort_values(by=5, ascending=False).groupby(10).apply(lambda x: x.iloc[0], include_groups=False)
    ctg_assign['midpoint'] = ctg_assign[[0,1]].mean(axis=1)
    ctg_assign = ctg_assign.sort_values(by=[9, 'midpoint'])

    Ctg_Assign[an] = ctg_assign
    
    tig_off_assembly = pd.DataFrame({i:len(seq.seq) for i,seq in assembly.items()}, index=['len']).T.loc[ctg_assign.index]
    
    tig_off_assembly['cumul_end'] = np.cumsum(tig_off_assembly['len'])
    tig_off_assembly['cumul_start'] = np.concatenate([np.array([0]), tig_off_assembly['cumul_end'].values[:-1]])
    Tig_Off_Assembly[an] = tig_off_assembly
    
    coords['sstart'] = coords[0]
    coords['send'] = coords[1]
    
    for ctg, df in coords.groupby(10):
        strand = ctg_assign.loc[ctg, 8]
        if strand == -1:
            coords.loc[df.index, 'qstart'] = np.abs(coords[2] - tig_off_assembly.loc[ctg, 'len'])
            coords.loc[df.index, 'qend'] = np.abs(coords[3] - tig_off_assembly.loc[ctg, 'len'])
    
        else:
            coords.loc[df.index, 'qstart'] = coords[2]
            coords.loc[df.index, 'qend'] = coords[3]
    coords['strand'] = np.where(coords['qstart'] < coords['qend'], '+', '-')
    
    coords['Qstart'] = coords['qstart'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
    coords['Qend'] = coords['qend'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
    
    coords['Sstart'] = coords['sstart'] + v.tig_off.loc[coords[9], 'cumul_start'].values
    coords['Send'] = coords['send'] + v.tig_off.loc[coords[9], 'cumul_start'].values
    
    Coords[an] = coords

# XhoI restriction digests

In [ ]:
from Bio import Restriction
for s in [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]:
    frag = []
    for sr in Assembly[f'{s}_canu_defaults'].values():
        frag.extend(list(Restriction.XhoI.catalyse(sr.seq)))
    
    d = np.log10(np.array([len(f) for f in frag]))
    #plt.plot(np.sort(d), np.linspace(0,1,d.shape[0]), label=s)
    plt.plot(np.sort(d), np.arange(0, d.shape[0]), label=s)

plt.legend()
plt.show()
plt.close()

In [ ]:
for an in assemblies.index:
        
    coords = Coords[an]
    tig_off_assembly = Tig_Off_Assembly[an]
    ctg_assign = Ctg_Assign[an]
    
    fig, ax = plt.subplots(figsize=[10, 10])
    
    for i in coords.index:
        ctg = coords.loc[i, 10]

        if ctg_assign.loc[ctg, 8] == -1:
            c = 'aqua'
        else:
            c = 'black'
        X = coords.loc[i, ['Sstart', 'Send']].values
        Y = coords.loc[i, ['Qstart', 'Qend']].values
        ax.plot(X, Y, c=c, lw=2, zorder=1)
    
    max_x = tig_off['cumul_end'].iloc[-1]
    max_y = tig_off_assembly['cumul_end'].iloc[-1]
    ax.set_xlim(0, max_x)
    ax.set_ylim(0, max_y)
    
    for ctg in tig_off_assembly.iloc[:-1].index:
        ax.axhline(tig_off_assembly.loc[ctg, 'cumul_end'], lw=0.5, zorder=0, c='k', ls=':')
        if tig_off_assembly.loc[ctg, 'len'] > 1e4:
            y1, y2 = tig_off_assembly.loc[ctg, ['cumul_start', 'cumul_end']]
            ax.text(max_x*1.02, np.mean([y1, y2]), ctg, va='center', size=6)
    
    for chrom in tig_off.iloc[:-1].index:
        x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']]
        ax.axvline(x1, lw=0.5, zorder=0, c='k', ls=':')
        ax.text(np.mean([x1, x2]), max_y*1.02, chr_ctg_to_alias[chrom], size=8, rotation=90)
        if chrom in centromeres.index:
            ax.axvline(centromeres.loc[chrom]+x1, lw=3, c='blue', alpha=0.2, zorder=0)
    
    xticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])
    xtick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])*1e-3
    ax.set_xticks(xticks)
    ax.set_xticklabels([f'{x:.0f}' if x>0 else '' for x in xtick_labels], size=8, rotation=90)
    ax.set_xlabel('S288C reference genome (kb)')
    
    yticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])
    ytick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])*1e-3
    ax.set_yticks(yticks)
    ax.set_yticklabels([f'{y:.0f}' if y>0 else '' for y in ytick_labels], size=8)
    ax.set_ylabel('Assembly (kb)')
    
    ax.set_title(an)
    
    plt.tight_layout()
    #plt.savefig(f'{fig_path}dotplots/dotplot_{an}.png', dpi=300)
    #plt.show()
    plt.close()

In [ ]:
# export reordered genomes
for an, ctg_assign in Ctg_Assign.items():
    new_genome = []
    for ctg in ctg_assign.index:
        
        seq = Assembly[an][ctg].seq
        orient = 'plus'
        if ctg_assign.loc[ctg, 8] == -1:
            seq = seq.reverse_complement()
            orient = 'minus'

        new_sr = SeqRecord.SeqRecord(seq=seq, id=f'{ctg}_{orient}', description='')
        new_genome.append(new_sr)

        #with open(f'{v.base_dir}data/reordered_assemblies/{an}_reordered.fasta', 'w') as handle:
        #    SeqIO.write(new_genome, handle, 'fasta')

In [ ]:
Assembly_Reordered = {}
Tig_Off_Assembly_Reordered = {}
Coords_Reordered = {}

for an in assemblies.index:
    
    assembly = [seq for seq in SeqIO.parse(f'{v.base_dir}data/reordered_assemblies/{an}_reordered.fasta', 'fasta')]
    Assembly_Reordered[an] = assembly

    tig_off_assembly = pd.DataFrame({seq.id:len(seq.seq) for seq in assembly}, index=['len']).T
    tig_off_assembly['cumul_end'] = np.cumsum(tig_off_assembly['len'])
    tig_off_assembly['cumul_start'] = np.concatenate([np.array([0]), tig_off_assembly['cumul_end'].values[:-1]])
    Tig_Off_Assembly_Reordered[an] = tig_off_assembly

    coords_path = f'{v.base_dir}mummer/{an}_reordered.S288C.coords'
    coords = pd.read_csv(coords_path, sep='\t', skiprows=4, header=None)
    coords[10] = coords[10].astype(str)
    coords = coords.loc[(coords[4]>5000) & (coords[5]>5000)]
    coords['sstart'] = coords[0]
    coords['send'] = coords[1]
    coords['qstart'] = coords[2]
    coords['qend'] = coords[3]
    coords['strand'] = np.where(coords['qstart'] < coords['qend'], '+', '-')
    
    coords['Qstart'] = coords['qstart'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
    coords['Qend'] = coords['qend'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
    
    coords['Sstart'] = coords['sstart'] + v.tig_off.loc[coords[9], 'cumul_start'].values
    coords['Send'] = coords['send'] + v.tig_off.loc[coords[9], 'cumul_start'].values
    
    Coords_Reordered[an] = coords

In [ ]:
Asm_N50 = {}
for an in assemblies.index:
    cl = np.sort(np.array([len(seq.seq) for seq in Assembly_Reordered[an]]))[::-1]
    n50 = cl[np.cumsum(cl) > 0.5*cl.sum()][0]
    Asm_N50[an] = n50
    assemblies.loc[an, 'N50'] = n50

In [ ]:
for an in assemblies.index:
        
    coords = Coords_Reordered[an]
    tig_off_assembly = Tig_Off_Assembly_Reordered[an]
    
    fig, ax = plt.subplots(figsize=[10, 10])
    
    for i in coords.index:
        ctg = coords.loc[i, 10]


        X = coords.loc[i, ['Sstart', 'Send']].values
        Y = coords.loc[i, ['Qstart', 'Qend']].values
        ax.plot(X, Y, c='k', lw=2, zorder=1)
    
    max_x = v.tig_off['cumul_end'].iloc[-1]
    max_y = tig_off_assembly['cumul_end'].iloc[-1]
    ax.set_xlim(0, max_x)
    ax.set_ylim(0, max_y)
    
    for ctg in tig_off_assembly.iloc[:-1].index:
        ax.axhline(tig_off_assembly.loc[ctg, 'cumul_end'], lw=0.5, zorder=0, c='k', ls=':')
        if tig_off_assembly.loc[ctg, 'len'] > 1e4:
            y1, y2 = tig_off_assembly.loc[ctg, ['cumul_start', 'cumul_end']]
            ax.text(max_x*1.02, np.mean([y1, y2]), ctg, va='center', size=6)
    
    for chrom in v.tig_off.iloc[:-1].index:
        x1, x2 = v.tig_off.loc[chrom, ['cumul_start', 'cumul_end']]
        ax.axvline(x1, lw=0.5, zorder=0, c='k', ls=':')
        ax.text(np.mean([x1, x2]), max_y*1.02, v.chr_ctg_to_alias[chrom], size=8, rotation=90)
        if chrom in v.centromeres.index:
            ax.axvline(v.centromeres.loc[chrom]+x1, lw=3, c='blue', alpha=0.2, zorder=0)
    
    xticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in v.tig_off.iloc[:-1].iterrows()])
    xtick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in v.tig_off.iloc[:-1].iterrows()])*1e-3
    ax.set_xticks(xticks)
    ax.set_xticklabels([f'{x:.0f}' if x>0 else '' for x in xtick_labels], size=8, rotation=90)
    ax.set_xlabel('S288C reference genome (kb)')
    
    yticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])
    ytick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])*1e-3
    ax.set_yticks(yticks)
    ax.set_yticklabels([f'{y:.0f}' if y>0 else '' for y in ytick_labels], size=8)
    ax.set_ylabel('Assembly (kb)')
    
    ax.set_title(f'{an} reordered')
    
    plt.tight_layout()
    #plt.savefig(f'{fig_path}dotplots/dotplot_{an}_reordered.png', dpi=300)
    plt.show()
    plt.close()

# Export reordered contigs trimmed to exclude subtelomeric regions for long read mapping

In [ ]:
asm_chrom_ends = pd.read_excel('assemblies_chrom_ends.xlsx')
asm_chrom_ends = asm_chrom_ends.loc[~asm_chrom_ends['start'].isna()].astype({'end':int, 'start':int, 'strand':int})

In [ ]:
for an, df in asm_chrom_ends.set_index(['chrom', 'chrom_end']).groupby('assembly_name'):
    
    if an == 'YJM948_canu_defaults':
    
        chrom_end_store = []
        with open(f'{v.base_dir}repeatmasker_asm/{an}_reordered.fasta.masked', 'r') as handle:
            assembly_reordered_dict = {seq.id:seq for seq in SeqIO.parse(handle, 'fasta')}
        #assembly_reordered_dict = {seq.id:seq for seq in Assembly_Reordered[an]}
        
        for chrom in range(1, 17):
    
            tigL, posL = df.loc[(chrom, 'L'), ['tig', 'end']]
            tigR, posR = df.loc[(chrom, 'R'), ['tig', 'end']]
            
            seq = None
            tig = None
            
            if all([tig in assembly_reordered_dict for tig in (tigL, tigR)]):
                # case when chromosome is a single contig
                if tigL == tigR:
                    seq = assembly_reordered_dict[tigL].seq[posL:posR]
                else:
                    seqL = assembly_reordered_dict[tigL].seq[posL:]
                    seqR = assembly_reordered_dict[tigR].seq[:posR]
                    seq = seqL + seqR
            else:
                if an == 'YJM955_canu_defaults' and chrom == 7:
                    tig = 'tig00000058_minus'
                    
                elif an == 'YJM965_canu_defaults' and chrom == 6:
                    tig = 'tig00000008_minus'
    
                else:
                    print(an, df.loc[chrom])
                seq = assembly_reordered_dict[tig].seq[posR:posL].reverse_complement()
            sr = SeqRecord.SeqRecord(seq, id=f'Chr{chrom}', description='')
            chrom_end_store.append(sr)
    
            with open(f'{v.base_dir}data/ref/{an}_chromEnds.fasta', 'w') as handle:
                SeqIO.write(chrom_end_store, handle, 'fasta')